# 05 - Model comparison: tree-based models vs. logistic regression

The baseline logistic regression plateaued around 65% accuracy / 0.68 F1, and we ruled out multicollinearity and class imbalance as the cause (see `04_baseline_logreg.ipynb`). The remaining suspect is that a *linear* model with hand-picked features structurally can't capture interaction effects — like the `material_diff` × `fullmove_number` interaction we discussed (a material lead means more late-game than early-game).

Tree-based models capture interactions and nonlinearity automatically, without us engineering them by hand — each split can implicitly condition on combinations of features. This notebook compares:
1. Logistic regression (reference, same as the `04` baseline)
2. Decision tree (single tree — intuitive, but prone to overfitting)
3. Random forest (many trees, averaged — reduces the overfitting variance of a single tree)
4. Histogram-based gradient boosting (`HistGradientBoostingClassifier` — sklearn's fast boosting implementation, well-suited to a dataset this size)

Note: **tree-based models don't need feature scaling** — a split like "is `material_diff` > 3?" doesn't care what units `material_diff` is in, unlike logistic regression's optimizer. We'll use the raw (unscaled) features for all tree models.

In [1]:
import sys
sys.path.append('..')

import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, classification_report,
)

from src.features import extract_features

train_df = pd.read_csv('../data/processed/train_features.csv', low_memory=False)
test_df = pd.read_csv('../data/processed/test_features.csv', low_memory=False)

feature_cols = list(extract_features(train_df['fen'].iloc[0]).keys())
X_train, y_train = train_df[feature_cols], train_df['white_win']
X_test, y_test = test_df[feature_cols], test_df['white_win']

X_train.shape, X_test.shape

((822936, 24), (208378, 24))

In [2]:
def evaluate_model(name, model, X_tr, X_te):
    start = time.time()
    model.fit(X_tr, y_train)
    fit_seconds = time.time() - start

    preds = model.predict(X_te)
    metrics = {
        'model': name,
        'accuracy': accuracy_score(y_test, preds),
        'precision': precision_score(y_test, preds),
        'recall': recall_score(y_test, preds),
        'f1': f1_score(y_test, preds),
        'recall_black_win': recall_score(y_test, preds, pos_label=0),
        'fit_seconds': fit_seconds,
    }

    print(f'{name}  (fit in {fit_seconds:.1f}s)')
    print(classification_report(y_test, preds, target_names=['black_win', 'white_win']))

    return metrics, model


results = []

## Model 1: Logistic regression (reference)
Same setup as `04_baseline_logreg.ipynb` — scaled features, all 24 columns, default L2. This is the number every tree model needs to beat to justify the extra complexity.

In [3]:
scaler = StandardScaler().fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

metrics, _ = evaluate_model('Logistic Regression', LogisticRegression(max_iter=1000), X_train_scaled, X_test_scaled)
results.append(metrics)

Logistic Regression  (fit in 0.4s)
              precision    recall  f1-score   support

   black_win       0.67      0.56      0.61    101264
   white_win       0.64      0.74      0.68    107114

    accuracy                           0.65    208378
   macro avg       0.65      0.65      0.65    208378
weighted avg       0.65      0.65      0.65    208378



## Model 2: Decision tree
An unconstrained tree will grow until every training leaf is pure — memorizing training data rather than learning general patterns (classic overfitting: check train accuracy vs. test accuracy below). Then compare against a depth-limited tree, which is the tree-model equivalent of what regularization does for logistic regression — constraining complexity to trade a bit of train-set fit for better generalization.

In [4]:
tree_unconstrained = DecisionTreeClassifier(random_state=42)
metrics, tree_unconstrained = evaluate_model('Decision Tree (unconstrained)', tree_unconstrained, X_train, X_test)
metrics['train_accuracy'] = accuracy_score(y_train, tree_unconstrained.predict(X_train))
results.append(metrics)

print(f"train accuracy: {metrics['train_accuracy']:.4f}  vs.  test accuracy: {metrics['accuracy']:.4f}")
print(f"tree depth: {tree_unconstrained.get_depth()}, leaves: {tree_unconstrained.get_n_leaves()}")

Decision Tree (unconstrained)  (fit in 3.7s)
              precision    recall  f1-score   support

   black_win       0.58      0.57      0.58    101264
   white_win       0.60      0.61      0.60    107114

    accuracy                           0.59    208378
   macro avg       0.59      0.59      0.59    208378
weighted avg       0.59      0.59      0.59    208378



train accuracy: 0.9400  vs.  test accuracy: 0.5908
tree depth: 57, leaves: 197822


In [5]:
tree_limited = DecisionTreeClassifier(max_depth=8, random_state=42)
metrics, tree_limited = evaluate_model('Decision Tree (max_depth=8)', tree_limited, X_train, X_test)
metrics['train_accuracy'] = accuracy_score(y_train, tree_limited.predict(X_train))
results.append(metrics)

print(f"train accuracy: {metrics['train_accuracy']:.4f}  vs.  test accuracy: {metrics['accuracy']:.4f}")

Decision Tree (max_depth=8)  (fit in 1.6s)
              precision    recall  f1-score   support

   black_win       0.69      0.50      0.58    101264
   white_win       0.63      0.79      0.70    107114

    accuracy                           0.65    208378
   macro avg       0.66      0.64      0.64    208378
weighted avg       0.66      0.65      0.64    208378

train accuracy: 0.6456  vs.  test accuracy: 0.6479


## Model 3: Random forest
Trains many decision trees, each on a bootstrapped sample with a random subset of features considered at each split, then averages their votes. Individual trees still overfit, but their errors are decorrelated by the randomness, so the average generalizes better than any single tree.

In [6]:
rf = RandomForestClassifier(n_estimators=200, max_depth=12, n_jobs=-1, random_state=42)
metrics, rf = evaluate_model('Random Forest', rf, X_train, X_test)
results.append(metrics)

Random Forest  (fit in 12.7s)
              precision    recall  f1-score   support

   black_win       0.69      0.53      0.60    101264
   white_win       0.63      0.77      0.70    107114

    accuracy                           0.65    208378
   macro avg       0.66      0.65      0.65    208378
weighted avg       0.66      0.65      0.65    208378



## Model 4: Gradient boosting
Rather than averaging independent trees like a random forest, boosting builds trees sequentially — each new tree specifically targets the previous ensemble's errors. `HistGradientBoostingClassifier` is sklearn's fast, histogram-based implementation (similar idea to LightGBM), well suited to a dataset this size.

In [7]:
hgb = HistGradientBoostingClassifier(max_iter=300, random_state=42)
metrics, hgb = evaluate_model('Gradient Boosting (HistGB)', hgb, X_train, X_test)
results.append(metrics)

Gradient Boosting (HistGB)  (fit in 7.3s)
              precision    recall  f1-score   support

   black_win       0.68      0.55      0.61    101264
   white_win       0.64      0.76      0.69    107114

    accuracy                           0.65    208378
   macro avg       0.66      0.65      0.65    208378
weighted avg       0.66      0.65      0.65    208378



## Compare all models

In [8]:
comparison = pd.DataFrame(results).set_index('model')
comparison[['accuracy', 'precision', 'recall', 'f1', 'recall_black_win', 'fit_seconds']].round(4)

,accuracy,precision,recall,f1,recall_black_win,fit_seconds
model,,,,,,
Logistic Regression,0.6493,0.6376,0.7363,0.6834,0.5573,0.4460
Decision Tree (unconstrained),0.5908,0.6005,0.6096,0.6050,0.5710,3.6830
Decision Tree (max_depth=8),0.6479,0.6253,0.7864,0.6966,0.5015,1.5618
Random Forest,0.6532,0.6330,0.7740,0.6965,0.5254,12.7313
Gradient Boosting (HistGB),0.6537,0.6377,0.7556,0.6917,0.5460,7.3273


## Feature importances
Random forest exposes `feature_importances_` directly (based on how much each feature reduces impurity across all its trees). `HistGradientBoostingClassifier` doesn't expose that natively, so we use permutation importance instead — shuffle one feature's values and see how much performance degrades, on a random subsample of test for speed. Compare both against the logistic regression coefficients from `04_baseline_logreg.ipynb` to see whether the models agree on what matters.

In [9]:
from sklearn.inspection import permutation_importance

sample_idx = X_test.sample(20000, random_state=42).index
perm = permutation_importance(
    hgb, X_test.loc[sample_idx], y_test.loc[sample_idx],
    n_repeats=5, random_state=42, n_jobs=-1,
)

importances = pd.DataFrame({
    'feature': feature_cols,
    'random_forest_importance': rf.feature_importances_,
    'gboost_permutation_importance': perm.importances_mean,
}).sort_values('random_forest_importance', ascending=False)

importances

,feature,random_forest_importance,gboost_permutation_importance
10,material_diff,0.438995,0.09434
18,mobility_diff,0.139154,0.01215
16,mobility_white,0.061341,0.00424
17,mobility_black,0.057039,0.00365
0,white_pawns,0.041482,0.01267
5,black_pawns,0.038149,0.00749
23,fullmove_number,0.027565,0.00970
9,black_queens,0.024247,0.00151
3,white_rooks,0.023183,0.00339
8,black_rooks,0.021730,0.00384
